In [1]:
!pip show sagemaker boto3

Name: sagemaker
Version: 2.251.0
Summary: Open source library for training and deploying models on Amazon SageMaker.
Home-page: https://github.com/aws/sagemaker-python-sdk
Author: Amazon Web Services
Author-email: 
License: 
Location: /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages
Requires: attrs, boto3, cloudpickle, docker, fastapi, google-pasta, graphene, importlib-metadata, jsonschema, numpy, omegaconf, packaging, pandas, pathos, platformdirs, protobuf, psutil, pyyaml, requests, sagemaker-core, schema, smdebug-rulesconfig, tblib, tqdm, urllib3, uvicorn
Required-by: 
---
Name: boto3
Version: 1.40.21
Summary: The AWS SDK for Python
Home-page: https://github.com/boto/boto3
Author: Amazon Web Services
Author-email: 
License: Apache License 2.0
Location: /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages
Requires: botocore, jmespath, s3transfer
Required-by: sagemaker, sagemaker-core


# LLaMA 3.1 on SageMaker

This notebook shows how to:

- Deploy a Hugging Face LLaMA 3.1 model on Amazon SageMaker  
- Run a quick test inference  
- **Always tear down the endpoint** to avoid idle GPU charges


## 0) Parameters

- Region defaults to the current AWS session, or `ap-southeast-2` if none is set  
- Endpoint resources (endpoint, config, model) will be **auto-cleaned** in a `finally` block


In [ ]:
import os, json, time, boto3, botocore, sagemaker
from sagemaker.huggingface import HuggingFaceModel

REGION = os.environ.get("AWS_REGION") or boto3.Session().region_name or "ap-southeast-2"
ROLE = sagemaker.get_execution_role()  # works on Notebook Instance

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ENDPOINT_NAME = "llama31-8b-endpoint"
INSTANCE_TYPE = "ml.g5.2xlarge"   # GPU cost, keep usage short!
AUTO_TIMEOUT_MIN = 20

IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Role:", ROLE)
print("Model:", MODEL_ID)
print("Endpoint:", ENDPOINT_NAME)
print("Image URI:", IMAGE_URI)